# **Statistical Rule-Based Baseline**

This notebook develops the statistical rule-based baseline for the HDFS_v1 anomaly detection study.

The purpose of the baseline model is to establish a simple, interpretable anomaly detection approach that can be compared against the supervised and unsupervised machine learning models.

Unlike the machine learning models, this approach does not learn complex patterns from the data. Instead, it uses statistical thresholds derived from normal training observations to identify behavior that falls outside the expected range.

For each engineered feature set, the baseline will:

1. Load the prepared training and testing datasets.
2. Use only normal training records to calculate feature thresholds.
3. Apply those thresholds to the test set.
4. Classify records as anomalous when one or more feature values exceed the learned thresholds.
5. Evaluate performance using Accuracy, Precision, Recall, and F1 Score.

The same baseline process will be applied to all four feature sets so that the effect of richer feature representations can be compared consistently.

In [14]:
# Import required libraries

from pathlib import Path

import pandas as pd

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [15]:
# Set project paths

current_path = Path.cwd()

if current_path.name == "notebooks":
    project_root = current_path.parent
else:
    project_root = current_path

splits_dir = project_root / "data" / "processed" / "splits"
results_dir = project_root / "results"

results_dir.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)
print("Splits folder:", splits_dir)
print("Results folder:", results_dir)

Project root: c:\Users\taman\Documents\hdfs-anomaly-detection-study
Splits folder: c:\Users\taman\Documents\hdfs-anomaly-detection-study\data\processed\splits
Results folder: c:\Users\taman\Documents\hdfs-anomaly-detection-study\results


## Load Prepared Training and Testing Data

The prepared train-test splits from the model preparation notebook are loaded for each feature set.

Each feature set contains:

- `X_train`: training features
- `X_test`: testing features
- `y_train`: training labels
- `y_test`: testing labels

The BlockId files are not required for calculating baseline performance, but they remain available in the split folders for later error analysis if needed.

In [16]:
# Load prepared train-test splits for each feature set

baseline_data = {}

for i in range(1, 5):

    feature_name = f"feature_set_{i}"
    feature_dir = splits_dir / feature_name

    baseline_data[feature_name] = {
        "X_train": pd.read_csv(feature_dir / "X_train.csv"),
        "X_test": pd.read_csv(feature_dir / "X_test.csv"),
        "y_train": pd.read_csv(feature_dir / "y_train.csv").squeeze(),
        "y_test": pd.read_csv(feature_dir / "y_test.csv").squeeze(),
    }

In [5]:
# Verify loaded dataset dimensions

for name, data in baseline_data.items():

    print(f"\n{name}")
    print(f"X_train: {data['X_train'].shape}")
    print(f"X_test : {data['X_test'].shape}")
    print(f"y_train: {data['y_train'].shape}")
    print(f"y_test : {data['y_test'].shape}")


feature_set_1
X_train: (460048, 29)
X_test : (115013, 29)
y_train: (460048,)
y_test : (115013,)

feature_set_2
X_train: (460048, 31)
X_test : (115013, 31)
y_train: (460048,)
y_test : (115013,)

feature_set_3
X_train: (460048, 311)
X_test : (115013, 311)
y_train: (460048,)
y_test : (115013,)

feature_set_4
X_train: (460048, 1082)
X_test : (115013, 1082)
y_train: (460048,)
y_test : (115013,)


## Calculate Statistical Thresholds from Normal Training Data

The statistical baseline is built by learning the range of normal behavior from the training data.

For each feature set, only records labeled as `0` are used to calculate thresholds. Since `0` represents successful HDFS block processing, these records define the expected behavior of normal system activity.

An IQR-based threshold is used for each feature:

- Q1: 25th percentile
- Q3: 75th percentile
- IQR: Q3 - Q1
- Upper threshold: Q3 + 1.5 × IQR

If a test record contains feature values above the learned upper threshold, that record may be classified as anomalous.

In [6]:
# Calculate IQR-based upper thresholds using only normal training records

baseline_thresholds = {}

for name, data in baseline_data.items():

    X_train = data["X_train"]
    y_train = data["y_train"]

    # Select only normal training records
    normal_train = X_train[y_train == 0]

    # Calculate Q1, Q3, and IQR for each feature
    q1 = normal_train.quantile(0.25)
    q3 = normal_train.quantile(0.75)
    iqr = q3 - q1

    # Calculate the upper threshold for each feature
    upper_threshold = q3 + (1.5 * iqr)

    baseline_thresholds[name] = upper_threshold

In [7]:
# Display threshold dimensions for each feature set

for name, thresholds in baseline_thresholds.items():
    print(f"{name}: {thresholds.shape[0]:,} thresholds")

feature_set_1: 29 thresholds
feature_set_2: 31 thresholds
feature_set_3: 311 thresholds
feature_set_4: 1,082 thresholds


The statistical thresholds define the expected range of normal behavior learned from the training data. A separate threshold is calculated for every engineered feature using only the normal training records. For example, one threshold is learned for the occurrence of each individual event (E1 through E29), while additional thresholds are learned for the engineered trace features and ordered event sequence features. Together, these thresholds describe the maximum values typically observed during normal HDFS block processing and will be used to identify unusually large feature values in unseen data.

## Review Learned Statistical Thresholds

Before applying the statistical baseline, the learned thresholds are examined to better understand the characteristics of normal system behavior.

Each threshold represents the largest value that would still be considered typical for a particular feature based on the normal training data. Because every engineered feature receives its own threshold, the number of learned thresholds matches the number of feature columns in each feature set.

The tables below display the learned thresholds for each feature set.

The statistical feature thresholds are calculated only from normal training records. After these thresholds are learned, each training record is scored by counting how many features exceed their normal threshold. Because the training labels are known, different violation-count cutoffs can be evaluated on the training set to determine how many threshold violations should be required before classifying a block as anomalous. The selected cutoff is then applied to the testing set for final evaluation.

In [12]:
# Display the final learned thresholds for each feature set

for name, thresholds in baseline_thresholds.items():

    print(f"\n{name}")

    display(thresholds.tail(10))


feature_set_1


E20    0.0
E21    3.0
E22    1.0
E23    3.0
E24    0.0
E25    0.0
E26    3.0
E27    0.0
E28    0.0
E29    0.0
dtype: float64


feature_set_2


E22                    1.0
E23                    3.0
E24                    0.0
E25                    0.0
E26                    3.0
E27                    0.0
E28                    0.0
E29                    0.0
trace_length          21.5
unique_event_count     9.5
dtype: float64


feature_set_3


E8->E14     0.0
E5->E1      0.0
E15->E12    0.0
E22->E12    0.0
E18->E21    0.0
E18->E2     0.0
E23->E26    0.0
E18->E11    0.0
E13->E18    0.0
E27->E27    0.0
dtype: float64


feature_set_4


E27->E26->E16    0.0
E5->E13->E9      0.0
E13->E13->E18    0.0
E26->E27->E27    0.0
E27->E27->E26    0.0
E3->E3->E11      0.0
E3->E26->E22     0.0
E22->E26->E13    0.0
E22->E13->E26    0.0
E9->E9->E27      0.0
dtype: float64

To verify that statistical thresholds were successfully generated for every engineered feature, the final learned thresholds from each feature set are displayed. Feature Sets 3 and 4 illustrate that thresholds were calculated not only for event occurrence counts and basic trace features, but also for the engineered two-event and three-event sequence features.

In [11]:
# Export learned statistical thresholds for each feature set

baseline_results_dir = results_dir / "statistical_baseline"
baseline_results_dir.mkdir(parents=True, exist_ok=True)

for name, thresholds in baseline_thresholds.items():

    threshold_path = baseline_results_dir / f"{name}_thresholds.csv"

    thresholds.to_csv(
        threshold_path,
        header=["upper_threshold"]
    )

    print(f"{name} thresholds saved to:")
    print(threshold_path)

feature_set_1 thresholds saved to:
c:\Users\taman\Documents\hdfs-anomaly-detection-study\results\statistical_baseline\feature_set_1_thresholds.csv
feature_set_2 thresholds saved to:
c:\Users\taman\Documents\hdfs-anomaly-detection-study\results\statistical_baseline\feature_set_2_thresholds.csv
feature_set_3 thresholds saved to:
c:\Users\taman\Documents\hdfs-anomaly-detection-study\results\statistical_baseline\feature_set_3_thresholds.csv
feature_set_4 thresholds saved to:
c:\Users\taman\Documents\hdfs-anomaly-detection-study\results\statistical_baseline\feature_set_4_thresholds.csv


## Count Threshold Violations for Each Training Block

The learned statistical thresholds have now been exported for each feature set. The next step is to determine the threshold violation count required to classify an HDFS block as anomalous.

To establish this violation cutoff, the learned statistical thresholds are applied to both the normal and anomalous training records. For each HDFS block, every engineered feature is compared against its corresponding statistical threshold, and each value that exceeds its threshold is counted as one violation. The resulting violation counts are then used to determine the optimal decision cutoff for classifying a block as anomalous.

In [17]:
# Count the number of threshold violations for each training block

# Create a dictionary to store the violation counts for each engineered feature set
training_violations = {}

# Process each feature set independently
for name, data in baseline_data.items():

    # Retrieve the training features
    X_train = data["X_train"]

    # Retrieve the learned statistical thresholds
    thresholds = baseline_thresholds[name]

    # Compare every feature value against its corresponding threshold Values exceeding the threshold are marked as True
    threshold_comparison = X_train > thresholds

    # Count the number of threshold violations for each HDFS block
    violation_counts = threshold_comparison.sum(axis=1)

    # Store the violation counts together with the corresponding ground-truth training labels
    training_violations[name] = pd.DataFrame({
        "Violation_Count": violation_counts,
        "Label": data["y_train"]
    })

# Display the first few violation counts for Feature Set 1

training_violations["feature_set_1"].head()

,Violation_Count,Label
0,0,0
1,3,0
2,0,0
3,1,1
4,3,0


In [18]:
# Summarize violation counts for normal and anomalous training records

for name, df in training_violations.items():

    print(f"\n{name}")

    # Summarize violation counts for normal blocks
    print("\nNormal Blocks")
    print(
        df[df["Label"] == 0]["Violation_Count"].describe()
    )

    # Summarize violation counts for anomalous blocks
    print("\nAnomalous Blocks")
    print(
        df[df["Label"] == 1]["Violation_Count"].describe()
    )


feature_set_1

Normal Blocks
count    446578.000000
mean          0.630403
std           1.087240
min           0.000000
25%           0.000000
50%           0.000000
75%           1.000000
max          11.000000
Name: Violation_Count, dtype: float64

Anomalous Blocks
count    13470.000000
mean         2.774981
std          3.176438
min          0.000000
25%          1.000000
50%          1.000000
75%          4.000000
max         15.000000
Name: Violation_Count, dtype: float64

feature_set_2

Normal Blocks
count    446578.000000
mean          0.960862
std           1.701984
min           0.000000
25%           0.000000
50%           0.000000
75%           1.000000
max          13.000000
Name: Violation_Count, dtype: float64

Anomalous Blocks
count    13470.000000
mean         3.459985
std          4.016781
min          0.000000
25%          1.000000
50%          1.000000
75%          6.000000
max         17.000000
Name: Violation_Count, dtype: float64

feature_set_3

Normal Blocks
co

The summary statistics provide an initial comparison of threshold violations between normal and anomalous training records for each engineered feature set. Across all four feature sets, anomalous HDFS blocks exhibit a **higher average number of threshold violations than normal blocks**, indicating that anomalous behavior is more likely to exceed the learned statistical thresholds. As progressively richer feature representations are introduced, the average violation counts increase for both classes, while the separation between the normal and anomalous distributions also becomes more pronounced. Although some overlap remains between the two classes, these results suggest that the number of threshold violations is a meaningful indicator of anomalous behavior and can be used to determine an optimal classification cutoff.

## Determine the Optimal Violation Count Cutoff

The statistical thresholds define whether an individual engineered feature exceeds its expected range of normal behavior. However, a final decision rule is still required to determine how many threshold violations are necessary before an HDFS block should be classified as anomalous.

Rather than selecting this violation cutoff arbitrarily, multiple candidate cutoff values are evaluated using the labeled training data for each feature set. Each candidate cutoff is used to classify the training blocks as either normal or anomalous, and the resulting predictions are compared with the ground-truth labels. The cutoff that produces the best classification performance is selected as the final decision threshold for that feature set before being applied to the testing data.

In [19]:
# Evaluate every possible violation-count cutoff for each feature set

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

# Create a dictionary to store the evaluation results
cutoff_results = {}

# Process each engineered feature set independently
for name, violations in training_violations.items():

    # Retrieve the true training labels
    y_true = violations["Label"]

    # Retrieve the violation counts for every HDFS block
    violation_counts = violations["Violation_Count"]

    # Test every unique observed violation count as a candidate cutoff
    candidate_cutoffs = sorted(violation_counts.unique())

    results = []

    for cutoff in candidate_cutoffs:

        # Classify a block as anomalous if its violation count meets or exceeds the cutoff
        y_pred = (violation_counts >= cutoff).astype(int)

        # Calculate classification performance
        results.append({
            "Cutoff": cutoff,
            "Accuracy": accuracy_score(y_true, y_pred),
            "Precision": precision_score(y_true, y_pred, zero_division=0),
            "Recall": recall_score(y_true, y_pred, zero_division=0),
            "F1": f1_score(y_true, y_pred, zero_division=0),
        })

    # Store all evaluated cutoffs for this feature set
    cutoff_results[name] = pd.DataFrame(results)

In [20]:
# Evaluate every possible violation-count cutoff for each feature set

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

# Create a dictionary to store the evaluation results
cutoff_results = {}

# Process each engineered feature set independently
for name, violations in training_violations.items():

    # Retrieve the true training labels
    y_true = violations["Label"]

    # Retrieve the violation counts for every HDFS block
    violation_counts = violations["Violation_Count"]

    # Test every unique observed violation count as a candidate cutoff
    candidate_cutoffs = sorted(violation_counts.unique())

    results = []

    for cutoff in candidate_cutoffs:

        # Classify a block as anomalous if its violation count meets or exceeds the cutoff
        y_pred = (violation_counts >= cutoff).astype(int)

        # Calculate classification performance
        results.append({
            "Cutoff": cutoff,
            "Accuracy": accuracy_score(y_true, y_pred),
            "Precision": precision_score(y_true, y_pred, zero_division=0),
            "Recall": recall_score(y_true, y_pred, zero_division=0),
            "F1": f1_score(y_true, y_pred, zero_division=0),
        })

    # Store all evaluated cutoffs for this feature set
    cutoff_results[name] = pd.DataFrame(results)

In [21]:
# Display the highest-performing cutoff for each feature set

for name, results in cutoff_results.items():

    best_result = results.sort_values(
        by="F1",
        ascending=False
    ).head(1)

    print(f"\n{name}")
    display(best_result)


feature_set_1


,Cutoff,Accuracy,Precision,Recall,F1
4,4,0.975887,0.722607,0.286414,0.410229



feature_set_2


,Cutoff,Accuracy,Precision,Recall,F1
6,6,0.975846,0.721617,0.285004,0.408622



feature_set_3


,Cutoff,Accuracy,Precision,Recall,F1
15,15,0.974633,0.67301,0.259911,0.375



feature_set_4


,Cutoff,Accuracy,Precision,Recall,F1
19,19,0.974974,0.716914,0.240089,0.359713


In [22]:
# Save the optimal cutoff and performance metrics for each feature set

optimal_cutoffs = []

for name, results in cutoff_results.items():

    # Select the cutoff with the highest F1 score
    best_result = results.sort_values(
        by="F1",
        ascending=False
    ).iloc[0]

    optimal_cutoffs.append({
        "Feature_Set": name,
        "Optimal_Cutoff": best_result["Cutoff"],
        "Accuracy": best_result["Accuracy"],
        "Precision": best_result["Precision"],
        "Recall": best_result["Recall"],
        "F1": best_result["F1"],
    })

optimal_cutoffs = pd.DataFrame(optimal_cutoffs)

display(optimal_cutoffs)

,Feature_Set,Optimal_Cutoff,Accuracy,Precision,Recall,F1
0,feature_set_1,4.0,0.975887,0.722607,0.286414,0.410229
1,feature_set_2,6.0,0.975846,0.721617,0.285004,0.408622
2,feature_set_3,15.0,0.974633,0.673010,0.259911,0.375000
3,feature_set_4,19.0,0.974974,0.716914,0.240089,0.359713


In [23]:
# Export the learned optimal violation cutoffs

optimal_cutoff_path = (
    baseline_results_dir / "optimal_violation_cutoffs.csv"
)

optimal_cutoffs.to_csv(
    optimal_cutoff_path,
    index=False
)

print("Optimal violation cutoffs saved to:")
print(optimal_cutoff_path)

Optimal violation cutoffs saved to:
c:\Users\taman\Documents\hdfs-anomaly-detection-study\results\statistical_baseline\optimal_violation_cutoffs.csv


The optimal violation count cutoff for each engineered feature set has now been learned from the training data. Rather than selecting a cutoff manually, every observed violation count was evaluated as a possible decision threshold. For each candidate cutoff, the training blocks were classified as either normal or anomalous and compared against the ground truth labels. The cutoff that produced the highest F1 score was selected as the final decision threshold for that feature set.

The results indicate that the optimal cutoff increases as additional engineered features are introduced. This trend is consistent with the earlier summary statistics, which showed that richer feature representations produce a greater number of threshold violations for both normal and anomalous blocks. Consequently, larger feature sets naturally require a higher violation count before a block should be classified as anomalous.

Although the average violation counts provided an initial indication of where the decision boundary might occur, the optimal cutoffs were learned directly from the labeled training data rather than being estimated from the descriptive statistics alone. This approach ensures that the selected decision threshold reflects the best observed classification performance instead of relying solely on the average behavior of each class.

The learned violation cutoffs are now fixed and will be applied to the unseen testing data to evaluate the final performance of the statistical rule based baseline.

## Evaluate Statistical Baseline on Testing Data

The learned statistical thresholds and optimal violation cutoffs are now applied to the unseen testing data.

For each feature set, the testing records are scored by counting how many engineered features exceed their learned thresholds. A block is classified as anomalous if its violation count is greater than or equal to the optimal cutoff selected from the training data.

This step provides the final test performance of the statistical rule based baseline.

In [24]:
# Apply the statistical baseline to the testing data

test_results = []

for name, data in baseline_data.items():

    # Retrieve testing features and labels
    X_test = data["X_test"]
    y_test = data["y_test"]

    # Retrieve learned thresholds and optimal cutoff
    thresholds = baseline_thresholds[name]

    cutoff = optimal_cutoffs.loc[
        optimal_cutoffs["Feature_Set"] == name,
        "Optimal_Cutoff"
    ].iloc[0]

    # Count threshold violations for each testing block
    test_violation_counts = (X_test > thresholds).sum(axis=1)

    # Classify blocks based on the learned violation cutoff
    y_pred = (test_violation_counts >= cutoff).astype(int)

    # Store final test performance metrics
    test_results.append({
        "Feature_Set": name,
        "Optimal_Cutoff": cutoff,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
    })

statistical_baseline_results = pd.DataFrame(test_results)

display(statistical_baseline_results)

,Feature_Set,Optimal_Cutoff,Accuracy,Precision,Recall,F1
0,feature_set_1,4.0,0.976203,0.744384,0.285333,0.412535
1,feature_set_2,6.0,0.976168,0.743590,0.284145,0.411171
2,feature_set_3,15.0,0.974838,0.686909,0.258610,0.375755
3,feature_set_4,19.0,0.975020,0.721180,0.239608,0.359706


In [25]:
# Export final statistical baseline test results

baseline_results_path = (
    baseline_results_dir / "statistical_baseline_test_results.csv"
)

statistical_baseline_results.to_csv(
    baseline_results_path,
    index=False
)

print("Statistical baseline test results saved to:")
print(baseline_results_path)

Statistical baseline test results saved to:
c:\Users\taman\Documents\hdfs-anomaly-detection-study\results\statistical_baseline\statistical_baseline_test_results.csv


## Statistical Baseline Results

The statistical rule based baseline was evaluated on the unseen testing data using the optimal violation count cutoff learned from the training data for each engineered feature set. The final results indicate that Feature Set 1 produced the strongest overall performance, achieving the highest F1 score while maintaining nearly identical accuracy to the remaining feature sets.

Although the richer feature sets introduced additional information through basic trace characteristics and sequential event patterns, these additions did not improve the statistical baseline. Instead, the F1 score gradually decreased as more engineered features were incorporated. This suggests that the larger number of statistical thresholds increased the total number of threshold violations for both normal and anomalous blocks, reducing the ability of the simple rule based decision process to distinguish between the two classes.

This result highlights an important distinction between feature engineering and model complexity. While richer feature representations often provide more information, they do not necessarily improve simpler statistical decision rules. Additional features may instead introduce greater variability and overlap between normal and anomalous behavior when each feature is evaluated independently against a fixed statistical threshold.

The statistical baseline established in this notebook serves as the reference model for the remainder of the study. The supervised Logistic Regression and Random Forest models, along with the unsupervised Isolation Forest model, will now be evaluated using the same engineered feature sets to determine whether machine learning methods can better leverage the additional sequential information introduced in Feature Sets 3 and 4.